# 🔧 Calliope AI Setup & Configuration

**Run this notebook FIRST** before using any other Calliope notebooks!

This notebook configures the Calliope magic commands to connect to your Data Agent API.

## What This Does

1. Loads the jupyter-ai extension
2. Configures the Calliope Data Agent API endpoint
3. Tests the connection
4. Verifies available datasources

## Prerequisites

Make sure the Data Agent is running:
```bash
docker-compose ps data-agent
# Should show as "running" (healthy)
```

## Step 1: Load Jupyter AI Extension

In [ ]:
%load_ext jupyter_ai

## Step 2: Configure API Endpoint

Set the base URL for the Calliope Data Agent API.

In [ ]:
import os

# Configuration options (choose one):

# Option 1: Running in Docker/JupyterHub environment
API_BASE_URL = os.environ.get('DATA_AGENT_URL', 'http://data-agent:5000')

# Option 2: Local development (uncomment if needed)
# API_BASE_URL = 'http://localhost:5000'

# Option 3: Custom URL (uncomment and modify if needed)
# API_BASE_URL = 'http://your-custom-host:5000'

# Set the environment variable for jupyter-ai
os.environ['CALLIOPE_API_BASE_URL'] = API_BASE_URL

print(f"✅ Calliope API configured: {API_BASE_URL}")

## Step 3: Test API Connection

In [ ]:
import requests
import json

def test_api_connection(base_url):
    """Test connection to Calliope Data Agent API"""
    print(f"Testing connection to: {base_url}")
    print("-" * 60)
    
    # Try multiple common endpoints
    endpoints_to_try = [
        '/api/datasources',
        '/api/health',
        '/api/status',
        '/health',
        '/status'
    ]
    
    for endpoint in endpoints_to_try:
        try:
            url = f"{base_url}{endpoint}"
            response = requests.get(url, timeout=5)
            
            if response.status_code == 200:
                print(f"✅ {endpoint}: OK")
                if 'datasources' in endpoint:
                    data = response.json()
                    print(f"   Found {len(data.get('datasources', []))} datasources")
                return True
            else:
                print(f"⚠️  {endpoint}: HTTP {response.status_code}")
        except requests.exceptions.Timeout:
            print(f"⏱️  {endpoint}: Timeout")
        except requests.exceptions.ConnectionError:
            print(f"❌ {endpoint}: Connection refused")
        except Exception as e:
            print(f"❌ {endpoint}: {str(e)[:50]}")
    
    return False

# Test the connection
if test_api_connection(API_BASE_URL):
    print("\n" + "=" * 60)
    print("✅ SUCCESS: Calliope API is accessible!")
    print("=" * 60)
else:
    print("\n" + "=" * 60)
    print("❌ ERROR: Could not connect to Calliope API")
    print("\nTroubleshooting:")
    print("1. Check if data-agent container is running:")
    print("   docker-compose ps data-agent")
    print("\n2. Check container logs:")
    print("   docker logs data-agent")
    print("\n3. Verify the API_BASE_URL is correct")
    print("=" * 60)

## Step 4: Verify Available Datasources

In [ ]:
try:
    response = requests.get(f"{API_BASE_URL}/api/datasources")
    response.raise_for_status()
    
    datasources = response.json()['datasources']
    
    print("📊 Available Datasources:")
    print("=" * 80)
    
    # Group by dialect
    mysql_ds = [ds for ds in datasources if ds['dialect'] == 'mysql']
    postgres_ds = [ds for ds in datasources if ds['dialect'] == 'postgres']
    
    if mysql_ds:
        print("\n🐬 MySQL Databases:")
        for ds in mysql_ds:
            print(f"  • {ds['id']:15} - {ds['name']}")
    
    if postgres_ds:
        print("\n🐘 PostgreSQL Databases:")
        for ds in postgres_ds:
            print(f"  • {ds['id']:15} - {ds['name']}")
    
    print("\n" + "=" * 80)
    print(f"Total: {len(datasources)} datasources available")
    print("=" * 80)
    
    # Store in environment for other notebooks
    os.environ['CALLIOPE_DATASOURCES'] = json.dumps([ds['id'] for ds in datasources])
    
except Exception as e:
    print(f"❌ Error fetching datasources: {e}")
    print("\nMake sure the API endpoint is correct and accessible.")

## Step 5: Test a Simple Query

Let's try a simple query to make sure everything works!

In [ ]:
# Test with a simple API call (not using magic commands yet)
try:
    # Get first available datasource
    response = requests.get(f"{API_BASE_URL}/api/datasources")
    datasources = response.json()['datasources']
    
    if datasources:
        test_ds = datasources[0]['id']
        print(f"Testing with datasource: {test_ds}")
        
        # Try a simple query
        test_payload = {
            "datasource_id": test_ds,
            "sql": "SELECT 1 as test",
            "model": "anthropic"
        }
        
        response = requests.post(
            f"{API_BASE_URL}/api/sql/run_sql",
            json=test_payload,
            headers={'Content-Type': 'application/json'},
            timeout=10
        )
        
        if response.status_code == 200:
            print("✅ Test query successful!")
            print(f"Response: {response.json()}")
        else:
            print(f"⚠️  Query returned status: {response.status_code}")
            print(f"Response: {response.text}")
    else:
        print("❌ No datasources available")
        
except Exception as e:
    print(f"❌ Test query failed: {e}")

## Step 6: Save Configuration

Save the configuration to a file so other notebooks can load it.

In [ ]:
import json
from pathlib import Path

config = {
    'api_base_url': API_BASE_URL,
    'configured_at': pd.Timestamp.now().isoformat()
}

config_file = Path.home() / '.calliope_config.json'
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ Configuration saved to: {config_file}")
print(f"\nOther notebooks will automatically load this configuration.")

## ✅ Setup Complete!

Your Calliope environment is now configured. You can now use:

### Individual Datasource Notebooks:
- `Datasource-Sakila.ipynb` - DVD rental store (MySQL)
- `Datasource-Chinook.ipynb` - Music store (PostgreSQL)
- `Datasource-Northwind.ipynb` - Business operations (PostgreSQL)
- `Datasource-Employees.ipynb` - HR database (MySQL)
- `Datasource-World.ipynb` - Geographic data (MySQL)
- `Datasource-AirportDB.ipynb` - Aviation data (MySQL)

### Feature Showcases:
- `MySQL-Features-Showcase.ipynb` - MySQL-specific features
- `PostgreSQL-Features-Showcase.ipynb` - PostgreSQL-specific features

### Tutorial Notebooks:
- `Calliope-GettingStarted.ipynb` - Learn the basics
- `Calliope-AdvancedQueries.ipynb` - Complex analytics
- And more!

## Troubleshooting

If you have issues:

1. **Check Data Agent Status**:
```bash
docker-compose ps data-agent
docker logs data-agent
```

2. **Test API Directly**:
```bash
curl http://localhost:5000/api/datasources
```

3. **Verify Network**:
```bash
docker exec -it <lab-container> ping data-agent
```

4. **Re-run This Notebook**: The configuration might have changed